In [ ]:
# ============================================================
# User path configuration
# ============================================================

from pathlib import Path

# Edit this if you are not running the notebook from the evaluation root.
# The root should contain folders like:
#   Replogle_K562_essential_pseudo_pairing_evaluation/
#   Replogle_RPE_pseudo_pairing_evaluation/
#   NormanWeissman2019_pseudo_pairing_evaluation/
ROOT_DIR = Path.cwd()

# Example:
# ROOT_DIR = Path("/ibex/user/chenj0i/Perturbation/evaluation")

# Selection table name expected under:
#   <dataset>_pseudo_pairing_evaluation/<group>/result_analysis/
SELECT_CSV_NAME = "selected_variants_TEMPLATE_EDIT_ME.csv"

print(f"ROOT_DIR = {ROOT_DIR}")
print(f"SELECT_CSV_NAME = {SELECT_CSV_NAME}")


In [ ]:
"""Shared plotting functions for forward MLP MSE/MAE scatter plots.

This notebook was converted from the standalone plotting script.
Edit the y-axis limits in the metric-specific cells below.
"""

from __future__ import annotations

import argparse
import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ============================================================
# Metric-specific configuration
# ============================================================

METRIC_NAME = "mse"
X_COLUMN = "mlp_forward__input_only_mse_mean"
Y_COLUMN = "mlp_forward__model_mse_mean"
REL_COLUMN = "mlp_forward__relative_mse_reduction_mean"

X_LABEL = "Input-only MSE"
Y_LABEL = "Forward model MSE"
PLOT_TITLE = "Forward prediction comparison: MSE"
SAVE_NAME = "forward_mse_scatter_reference_shading"
REL_LABEL = "MSE reduction"

# ============================================================
# Paths
# ============================================================

ROOT_DIR = Path(os.getcwd())

DATASET_NAMES = [
    "Replogle_K562_essential",
    "Replogle_RPE",
    "NormanWeissman2019",
    "ChangYe",
    "ZhaoSims2021",
]

GROUPS = ["single", "dual", "multi"]

SELECT_CSV_NAME = "selected_variants_TEMPLATE_EDIT_ME.csv"
OUTPUT_SUBDIR = "/ibex/user/chenj0i/Perturbation/evaluation/scatter_plots"

# ============================================================
# CLEAR AXIS LIMIT CONTROLS
# ============================================================
# Y_LIM controls the y-axis only: forward model MSE.
#
# Accepted forms:
#   Y_LIM = None
#       -> Auto y-limit from selected rows plus the S0 reference.
#
#   Y_LIM = (0.0, 0.050)
#       -> Force y-axis from 0.0 to 0.050.
#
#   Y_LIM = (None, 0.050)
#       -> Auto lower bound, fixed upper bound 0.050.
#
#   Y_LIM = (0.0, None)
#       -> Fixed lower bound 0.0, auto upper bound.
#
# Use this when different datasets need different y-axis scales:
#   Y_LIM_BY_DATASET_GROUP = {
#       ("Replogle_K562_essential", "single"): (0.0, 0.050),
#       ("Replogle_RPE", "single"): (0.0, 0.040),
#   }
#
# Keep None first, run once, check the printed "[Axis]" values,
# then manually set fixed limits if you want matched panels.
Y_LIM = None  # type: Optional[Tuple[Optional[float], Optional[float]]]

Y_LIM_BY_DATASET_GROUP = {}  # type: Dict[Tuple[str, str], Tuple[Optional[float], Optional[float]]]

# X_LIM controls the x-axis only: input-only MSE.
X_LIM = None  # type: Optional[Tuple[Optional[float], Optional[float]]]

X_LIM_BY_DATASET_GROUP = {}  # type: Dict[Tuple[str, str], Tuple[Optional[float], Optional[float]]]

# If True, automatic lower limits start at zero.
# For error metrics this is usually preferred.
AUTO_LOWER_AT_ZERO = True

# Margin used only for automatic limits or None entries within X_LIM/Y_LIM.
AXIS_MARGIN_FRACTION = 0.05

# ============================================================
# Strategy labels and colors
# ============================================================

STRATEGY_PLOT_LABELS = {
    "S0_naive_mean_control_reference": "Naive\nmean\ncontrol",
    "S1_random_single_control": "Random\nsingle\ncontrol",
    "S2_random_average_controls": "Random\naverage\ncontrol",
    "S4_SEACell_balanced_random_sample": "Metacell\nbalanced\nrandom",
    "S3_SEACell_metacell_average": "Random\nmetacell\naverage",
    "S5_SEACell_OT_sampled_average": "Metacell OT\nsampled\naverage",
}

STRATEGY_BASE_COLORS = {
    "S0_naive_mean_control_reference": "#D0E0EF",
    "S1_random_single_control": "#6E8FB2",
    "S2_random_average_controls": "#7DA494",
    "S3_SEACell_metacell_average": "#E5A79A",
    "S4_SEACell_balanced_random_sample": "#EAB67A",
    "S5_SEACell_OT_sampled_average": "#9F8DB8",
}

S5_VARIANT_COLORS = {
    "200&5": "#D49AB5",
    "350&5": "#9F8DB8",
    "500&5": "#B66699",
}

# ============================================================
# Plot style controls
# ============================================================

matplotlib.rcParams["svg.fonttype"] = "none"
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
matplotlib.rcParams["axes.linewidth"] = 0.8
matplotlib.rcParams["xtick.major.width"] = 0.8
matplotlib.rcParams["ytick.major.width"] = 0.8

FIGSIZE = (10, 10)

POINT_SIZE = 175
REFERENCE_POINT_SIZE = 175
POINT_EDGE_COLOR = "black"
POINT_EDGE_WIDTH = 0.8

NAIVE_REFERENCE_COLOR = "#EE5862"
NAIVE_REFERENCE_LINESTYLE = "--"
NAIVE_REFERENCE_LINEWIDTH = 1.5

DIAGONAL_COLOR = "#222222"
DIAGONAL_LINESTYLE = ":"
DIAGONAL_LINEWIDTH = 0.8

UPPER_SHADE_FACE = "#DBDBDB"
UPPER_SHADE_ALPHA = 0.25
UPPER_SHADE_HATCH = "///"

RIGHT_SHADE_FACE = "#DBDBDB"
RIGHT_SHADE_ALPHA = 0.25
RIGHT_SHADE_HATCH = r"\\\\"

LEGEND_NCOL = 2
LEGEND_FONT_SIZE = 10
LEGEND_TITLE_FONT_SIZE = 9.0
LEGEND_Y_ANCHOR = -0.12
LEGEND_INCLUDE_REFERENCE_ITEMS = True
BOTTOM_MARGIN = 0.20

SHOW_REFERENCE_POINT = True
SHOW_GRID = False
SAVE_PNG = True
SAVE_SVG = True

DEOVERLAP_POINTS = True
MIN_POINT_DISTANCE_FRACTION = 0.015
POINT_REPEL_ITERATIONS = 10
SHOW_DISPLACEMENT_CONNECTORS = True


# ============================================================
# Helper functions
# ============================================================

def as_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y"])


def is_valid_color(x: Any) -> bool:
    if pd.isna(x):
        return False
    x = str(x).strip()
    return bool(x) and x.lower() not in {"nan", "none", "null"}


def clean_label_for_legend(label: str) -> str:
    return " ".join(str(label).replace("\n", " ").split())


def extract_variant_suffix(display_label: str) -> str:
    display_label = str(display_label)
    if "(" in display_label and ")" in display_label:
        return display_label[display_label.rfind("("):].strip()
    return ""


def build_plot_label(row: pd.Series) -> str:
    strategy = str(row.get("strategy", ""))
    base = STRATEGY_PLOT_LABELS.get(strategy, strategy)

    display_label = str(row.get("display_variant_label", ""))
    suffix = extract_variant_suffix(display_label)

    if suffix:
        return f"{base}\n{suffix}"
    return base


def choose_color(row: pd.Series) -> str:
    for col in ["manual_color", "color"]:
        if col in row.index and is_valid_color(row[col]):
            return str(row[col]).strip()

    strategy = str(row.get("strategy", ""))
    display_label = str(row.get("display_variant_label", ""))

    if strategy == "S5_SEACell_OT_sampled_average":
        for key, color in S5_VARIANT_COLORS.items():
            if key in display_label:
                return color

    return STRATEGY_BASE_COLORS.get(strategy, "#999999")


def format_relative_value(v: Any) -> str:
    if pd.isna(v):
        return "NA"
    v = float(v)
    if abs(v) <= 1.5:
        return f"{100 * v:.1f}%"
    return f"{v:.1f}%"


def get_reference_row(df_all: pd.DataFrame) -> pd.Series:
    ref = df_all[df_all["strategy"].astype(str) == "S0_naive_mean_control_reference"].copy()
    if ref.empty:
        raise ValueError("Cannot find S0_naive_mean_control_reference in the selection table.")
    return ref.iloc[0]


def get_selected_plot_df(df_all: pd.DataFrame) -> pd.DataFrame:
    if "select_for_final" not in df_all.columns:
        raise KeyError("The selection table must contain 'select_for_final'.")

    out = df_all[as_bool_series(df_all["select_for_final"])].copy()
    out = out[out["strategy"].astype(str) != "S0_naive_mean_control_reference"].copy()
    return out


def get_dataset_group_from_sub_path(sub_path: Path) -> Tuple[str, str]:
    dataset = sub_path.parent.name.replace("_pseudo_pairing_evaluation", "")
    group = sub_path.name
    return dataset, group


def get_dataset_group_title(sub_path: Path) -> str:
    dataset, group = get_dataset_group_from_sub_path(sub_path)
    return f"{dataset} | {group}"


def build_strategy_legend_label(row: pd.Series) -> str:
    label = clean_label_for_legend(row.get("plot_label", build_plot_label(row)))
    if REL_COLUMN in row.index and pd.notna(row[REL_COLUMN]):
        label += f"  |  {REL_LABEL}: {format_relative_value(row[REL_COLUMN])}"
    return label


def choose_axis_limit(
    values: np.ndarray,
    ref_value: float,
    user_limit,
    margin_fraction: float = AXIS_MARGIN_FRACTION,
    auto_lower_at_zero: bool = AUTO_LOWER_AT_ZERO,
) -> Tuple[float, float]:
    finite_values = np.asarray(values, dtype=float)
    finite_values = finite_values[np.isfinite(finite_values)]

    candidates = list(finite_values) + [float(ref_value)]
    candidates = [v for v in candidates if np.isfinite(v)]

    if not candidates:
        raise ValueError("Cannot determine axis limit because all values are non-finite.")

    data_min = min(candidates)
    data_max = max(candidates)

    if auto_lower_at_zero:
        auto_lower = 0.0
    else:
        auto_lower = max(0.0, data_min * (1.0 - margin_fraction))

    auto_upper = data_max * (1.0 + margin_fraction)
    if auto_upper <= auto_lower:
        auto_upper = auto_lower + 1e-9

    if user_limit is None:
        return float(auto_lower), float(auto_upper)

    if len(user_limit) != 2:
        raise ValueError("Axis limit must be None or a 2-tuple, e.g. (0.0, 0.05).")

    lower, upper = user_limit

    lower = auto_lower if lower is None else float(lower)
    upper = auto_upper if upper is None else float(upper)

    if not np.isfinite(lower) or not np.isfinite(upper):
        raise ValueError(f"Axis limit contains non-finite value: {user_limit}")
    if upper <= lower:
        raise ValueError(f"Axis upper limit must be greater than lower limit: {user_limit}")

    return lower, upper


def get_user_axis_limit(
    base_limit,
    per_group_limits,
    sub_path: Path,
):
    dataset, group = get_dataset_group_from_sub_path(sub_path)
    return per_group_limits.get((dataset, group), base_limit)


def deoverlap_xy(
    x_values: np.ndarray,
    y_values: np.ndarray,
    x_min: float,
    x_max: float,
    y_min: float,
    y_max: float,
    min_dist_fraction: float,
    n_iter: int,
    padding_fraction: float = 0.008,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    De-overlap close points by moving them only along the y-axis.
    The x coordinates remain unchanged.
    """
    x_values = np.asarray(x_values, dtype=float)
    y_values = np.asarray(y_values, dtype=float)

    n = len(x_values)
    if n <= 1:
        return x_values.copy(), y_values.copy()

    x_span = max(float(x_max - x_min), 1e-12)
    y_span = max(float(y_max - y_min), 1e-12)

    x_norm = (x_values - x_min) / x_span
    y_norm = (y_values - y_min) / y_span
    y_adj = y_norm.copy()

    min_dist = float(min_dist_fraction)
    pad = float(padding_fraction)

    def direction(i: int, j: int) -> float:
        return 1.0 if (i + j) % 2 == 0 else -1.0

    for _ in range(int(n_iter)):
        max_push = 0.0

        for i in range(n - 1):
            for j in range(i + 1, n):
                dx = float(x_norm[j] - x_norm[i])
                dy = float(y_adj[j] - y_adj[i])

                dist = float(np.sqrt(dx * dx + dy * dy))
                if dist >= min_dist:
                    continue
                if abs(dx) >= min_dist:
                    continue

                required_abs_dy = np.sqrt(max(min_dist * min_dist - dx * dx, 0.0))
                current_abs_dy = abs(dy)

                if current_abs_dy >= required_abs_dy:
                    continue

                push_total = required_abs_dy - current_abs_dy
                push_each = 0.5 * push_total

                sign = direction(i, j) if abs(dy) < 1e-12 else (1.0 if dy > 0 else -1.0)

                y_adj[i] -= sign * push_each
                y_adj[j] += sign * push_each

                max_push = max(max_push, push_each)

        y_adj = np.clip(y_adj, pad, 1.0 - pad)

        if max_push < 1e-5:
            break

    x_out = x_values.copy()
    y_out = y_min + y_adj * y_span

    return x_out, y_out


def add_reference_shadings(
    ax,
    ref_x: float,
    ref_y: float,
    x_lim,
    y_lim,
) -> None:
    """Shade regions worse than the S0 naive reference.

    Upper region:
        y > S0 forward-model error

    Right region:
        x > S0 input-only error
    """
    x_min, x_max = x_lim
    y_min, y_max = y_lim

    if ref_y < y_max:
        ax.axhspan(
            ref_y,
            y_max,
            xmin=0,
            xmax=1,
            facecolor=UPPER_SHADE_FACE,
            alpha=UPPER_SHADE_ALPHA,
            hatch=UPPER_SHADE_HATCH,
            edgecolor="#999999",
            linewidth=0.0,
            zorder=-3,
        )

    if ref_x < x_max:
        ax.axvspan(
            ref_x,
            x_max,
            ymin=0,
            ymax=1,
            facecolor=RIGHT_SHADE_FACE,
            alpha=RIGHT_SHADE_ALPHA,
            hatch=RIGHT_SHADE_HATCH,
            edgecolor="#999999",
            linewidth=0.0,
            zorder=-2,
        )


def make_bottom_legend(ax, df_plot: pd.DataFrame, ref_x: float, ref_y: float):
    handles = []  # type: List[Any]
    labels = []  # type: List[str]

    for _, row in df_plot.iterrows():
        handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="None",
                markersize=8.5,
                markerfacecolor=row["plot_color"],
                markeredgecolor=POINT_EDGE_COLOR,
                markeredgewidth=POINT_EDGE_WIDTH,
            )
        )
        labels.append(build_strategy_legend_label(row))

    if SHOW_REFERENCE_POINT:
        handles.append(
            Line2D(
                [0],
                [0],
                marker="X",
                linestyle="None",
                markersize=8.5,
                markerfacecolor="#4D4D4D",
                markeredgecolor="#4D4D4D",
                markeredgewidth=POINT_EDGE_WIDTH,
            )
        )
        labels.append("Naive average control")

    if LEGEND_INCLUDE_REFERENCE_ITEMS:
        handles.extend(
            [
                Line2D(
                    [0],
                    [0],
                    color=NAIVE_REFERENCE_COLOR,
                    linestyle=NAIVE_REFERENCE_LINESTYLE,
                    linewidth=NAIVE_REFERENCE_LINEWIDTH,
                ),
                Line2D(
                    [0],
                    [0],
                    color=DIAGONAL_COLOR,
                    linestyle=DIAGONAL_LINESTYLE,
                    linewidth=DIAGONAL_LINEWIDTH,
                ),
                Patch(
                    facecolor=UPPER_SHADE_FACE,
                    edgecolor="#999999",
                    alpha=UPPER_SHADE_ALPHA,
                    hatch=UPPER_SHADE_HATCH,
                    linewidth=0.0,
                ),
                Patch(
                    facecolor=RIGHT_SHADE_FACE,
                    edgecolor="#999999",
                    alpha=RIGHT_SHADE_ALPHA,
                    hatch=RIGHT_SHADE_HATCH,
                    linewidth=0.0,
                ),
            ]
        )
        labels.extend(
            [
                f"Naive reference corner: x={ref_x:.4g}, y={ref_y:.4g}",
                "y = x, no model improvement",
                "Model error above Naive reference",
                "Input-only error above Naive reference",
            ]
        )

    return ax.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, LEGEND_Y_ANCHOR),
        ncol=LEGEND_NCOL,
        frameon=False,
        fontsize=LEGEND_FONT_SIZE,
        title="Selected strategies and references",
        title_fontsize=LEGEND_TITLE_FONT_SIZE,
        handlelength=1.5,
        columnspacing=1.5,
        handletextpad=0.6,
        borderaxespad=0.0,
    )


def plot_forward_metric_scatter(
    df_all: pd.DataFrame,
    sub_path: Path,
    save_dir=None,
    figsize=FIGSIZE,
) -> Tuple[plt.Figure, plt.Axes]:
    required_cols = ["strategy", X_COLUMN, Y_COLUMN]
    missing_cols = [c for c in required_cols if c not in df_all.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    df_ref = get_reference_row(df_all)
    df_plot = get_selected_plot_df(df_all)

    df_plot[X_COLUMN] = pd.to_numeric(df_plot[X_COLUMN], errors="coerce")
    df_plot[Y_COLUMN] = pd.to_numeric(df_plot[Y_COLUMN], errors="coerce")
    if REL_COLUMN in df_plot.columns:
        df_plot[REL_COLUMN] = pd.to_numeric(df_plot[REL_COLUMN], errors="coerce")

    df_plot = df_plot.dropna(subset=[X_COLUMN, Y_COLUMN]).copy().reset_index(drop=True)

    if df_plot.empty:
        raise ValueError(f"No selected rows available for {METRIC_NAME} scatter plot.")

    ref_x = float(pd.to_numeric(pd.Series([df_ref[X_COLUMN]]), errors="coerce").iloc[0])
    ref_y = float(pd.to_numeric(pd.Series([df_ref[Y_COLUMN]]), errors="coerce").iloc[0])
    if not np.isfinite(ref_x) or not np.isfinite(ref_y):
        raise ValueError(f"S0 reference row has invalid values for {X_COLUMN} or {Y_COLUMN}.")

    df_plot["plot_color"] = df_plot.apply(choose_color, axis=1)
    df_plot["plot_label"] = df_plot.apply(build_plot_label, axis=1)

    user_x_lim = get_user_axis_limit(X_LIM, X_LIM_BY_DATASET_GROUP, sub_path)
    user_y_lim = get_user_axis_limit(Y_LIM, Y_LIM_BY_DATASET_GROUP, sub_path)

    x_lim = choose_axis_limit(
        values=df_plot[X_COLUMN].to_numpy(dtype=float),
        ref_value=ref_x,
        user_limit=user_x_lim,
    )
    y_lim = choose_axis_limit(
        values=df_plot[Y_COLUMN].to_numpy(dtype=float),
        ref_value=ref_y,
        user_limit=user_y_lim,
    )

    dataset_group_title = get_dataset_group_title(sub_path)
    print(f"[Axis] {dataset_group_title} | X_LIM used = {x_lim} | Y_LIM used = {y_lim}")

    x_raw = df_plot[X_COLUMN].to_numpy(dtype=float)
    y_raw = df_plot[Y_COLUMN].to_numpy(dtype=float)

    if DEOVERLAP_POINTS:
        x_draw, y_draw = deoverlap_xy(
            x_values=x_raw,
            y_values=y_raw,
            x_min=x_lim[0],
            x_max=x_lim[1],
            y_min=y_lim[0],
            y_max=y_lim[1],
            min_dist_fraction=MIN_POINT_DISTANCE_FRACTION,
            n_iter=POINT_REPEL_ITERATIONS,
        )
        # x is intentionally kept unchanged, matching the notebook behavior.
        x_draw = x_raw.copy()
    else:
        x_draw, y_draw = x_raw.copy(), y_raw.copy()

    df_plot["_x_draw"] = x_draw
    df_plot["_y_draw"] = y_draw

    fig, ax = plt.subplots(figsize=figsize)

    add_reference_shadings(ax, ref_x=ref_x, ref_y=ref_y, x_lim=x_lim, y_lim=y_lim)

    # S0 reference guides.
    ax.plot(
        [x_lim[0], x_lim[1]],
        [ref_y, ref_y],
        color=NAIVE_REFERENCE_COLOR,
        linestyle=NAIVE_REFERENCE_LINESTYLE,
        linewidth=NAIVE_REFERENCE_LINEWIDTH,
        alpha=0.75,
        zorder=2,
    )
    ax.plot(
        [ref_x, ref_x],
        [y_lim[0], ref_y],
        color="grey",
        linestyle=NAIVE_REFERENCE_LINESTYLE,
        linewidth=NAIVE_REFERENCE_LINEWIDTH,
        alpha=0.75,
        zorder=2,
    )

    if SHOW_REFERENCE_POINT:
        ax.scatter(
            ref_x,
            ref_y,
            s=REFERENCE_POINT_SIZE,
            marker="X",
            color="#4D4D4D",
            edgecolor=POINT_EDGE_COLOR,
            linewidth=POINT_EDGE_WIDTH,
            zorder=6,
            alpha=0.7,
        )

    # Diagonal y=x. Draw only over the common visible interval.
    diag_min = max(x_lim[0], y_lim[0])
    diag_max = min(x_lim[1], y_lim[1])
    if diag_max > diag_min:
        ax.plot(
            [diag_min, diag_max],
            [diag_min, diag_max],
            color=DIAGONAL_COLOR,
            linestyle=DIAGONAL_LINESTYLE,
            linewidth=DIAGONAL_LINEWIDTH,
            alpha=0.85,
            zorder=1,
        )

    if DEOVERLAP_POINTS and SHOW_DISPLACEMENT_CONNECTORS:
        norm_displacement = np.sqrt(
            ((x_draw - x_raw) / max(x_lim[1] - x_lim[0], 1e-12)) ** 2
            + ((y_draw - y_raw) / max(y_lim[1] - y_lim[0], 1e-12)) ** 2
        )

        for i in range(len(df_plot)):
            if norm_displacement[i] <= 1e-4:
                continue

            ax.plot(
                [x_raw[i], x_draw[i]],
                [y_raw[i], y_draw[i]],
                color="#777777",
                linewidth=0.6,
                alpha=0.45,
                zorder=3,
            )

    for _, row in df_plot.iterrows():
        ax.scatter(
            float(row["_x_draw"]),
            float(row["_y_draw"]),
            s=POINT_SIZE,
            color=row["plot_color"],
            edgecolor=POINT_EDGE_COLOR,
            linewidth=POINT_EDGE_WIDTH,
            zorder=5,
            alpha=0.98,
        )

    ax.set_xlim(*x_lim)
    ax.set_ylim(*y_lim)

    # Equal visual scaling only when x/y ranges are equal.
    # This avoids distorted/empty panels when you manually set a different Y_LIM.
    if np.isclose(x_lim[1] - x_lim[0], y_lim[1] - y_lim[0]):
        ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel(X_LABEL, fontsize=15)
    ax.set_ylabel(Y_LABEL, fontsize=15)
    ax.set_title(PLOT_TITLE, fontsize=18, weight="bold")

    if SHOW_GRID:
        ax.grid(True, linewidth=0.5, alpha=0.23, zorder=-5)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.text(
        0.02,
        0.95,
        dataset_group_title,
        transform=ax.transAxes,
        fontsize=15,
        color="#555555",
        ha="left",
        va="bottom",
    )

    make_bottom_legend(ax, df_plot=df_plot, ref_x=ref_x, ref_y=ref_y)

    fig.subplots_adjust(bottom=BOTTOM_MARGIN)

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        if SAVE_PNG:
            png_path = save_dir / f"{SAVE_NAME}.png"
            fig.savefig(png_path, dpi=300, bbox_inches="tight")
            print(f"[Saved] {png_path}")

        if SAVE_SVG:
            svg_path = save_dir / f"{SAVE_NAME}.svg"
            fig.savefig(svg_path, bbox_inches="tight")
            print(f"[Saved] {svg_path}")

    plt.close(fig)
    return fig, ax


def discover_valid_paths(root_dir: Path) -> List[Path]:
    candidate_paths = [root_dir / f"{name}_pseudo_pairing_evaluation" for name in DATASET_NAMES]
    detailed_sub_paths = [sub_path / group for sub_path in candidate_paths for group in GROUPS]
    return [sub_path for sub_path in detailed_sub_paths if sub_path.exists()]


def run_all(root_dir: Path) -> None:
    valid_paths = discover_valid_paths(root_dir)
    if not valid_paths:
        raise FileNotFoundError(
            f"No dataset/group folders found under {root_dir}. "
            "Check ROOT_DIR, DATASET_NAMES, and GROUPS."
        )

    for sub_path in valid_paths:
        select_csv = sub_path / "result_analysis" / SELECT_CSV_NAME
        if not select_csv.exists():
            print(f"[Skip] Missing selection table: {select_csv}")
            continue

        print(f"[Read] {select_csv}")
        df = pd.read_csv(select_csv)

        save_dir = OUTPUT_SUBDIR / sub_path
        plot_forward_metric_scatter(df, sub_path=sub_path, save_dir=save_dir)





In [ ]:
# ============================================================
# MSE plot settings
# ============================================================
# Y_LIM_MSE controls ONLY the y-axis: Forward model MSE.
#
# Examples:
#   Y_LIM_MSE = None
#   Y_LIM_MSE = (0.0, 0.050)
#   Y_LIM_MSE = (None, 0.050)
#   Y_LIM_MSE = (0.0, None)

Y_LIM_MSE = None
X_LIM_MSE = None

# Optional dataset/group-specific axis limits.
# Keys are: (dataset_name, group)
# group is usually one of: "single", "dual", "multi"
Y_LIM_BY_DATASET_GROUP_MSE = {
    ("Replogle_K562_essential", "single"): (0.10, 0.25),
    ("Replogle_RPE", "single"): (0.10, 0.30),
    ("NormanWeissman2019", "single"): (0.07, 0.16),
    ("NormanWeissman2019", "dual"): (0.07, 0.17),
    ("ChangYe", "single"): (0.05, 0.15),
    ("ZhaoSims2021", "single"): (0.10, 0.30),
}

X_LIM_BY_DATASET_GROUP_MSE = {
    ("Replogle_K562_essential", "single"): (0.10, 0.30),
    ("Replogle_RPE", "single"): (0.10, 0.35),
    ("NormanWeissman2019", "single"): (0.07, 0.20),
    ("NormanWeissman2019", "dual"): (0.07, 0.20),
    ("ChangYe", "single"): (0.10, 0.25),
    ("ZhaoSims2021", "single"): (0.10, 0.40),
}


In [ ]:
# ============================================================
# Run MSE plots
# ============================================================

METRIC_NAME = "mse"
X_COLUMN = "mlp_forward__input_only_mse_mean"
Y_COLUMN = "mlp_forward__model_mse_mean"
REL_COLUMN = "mlp_forward__relative_mse_reduction_mean"

X_LABEL = "Input-only MSE"
Y_LABEL = "Forward model MSE"
PLOT_TITLE = "Forward prediction comparison: MSE"
SAVE_NAME = "forward_mse_scatter_reference_shading"
REL_LABEL = "MSE reduction"

Y_LIM = Y_LIM_MSE
X_LIM = X_LIM_MSE
Y_LIM_BY_DATASET_GROUP = Y_LIM_BY_DATASET_GROUP_MSE
X_LIM_BY_DATASET_GROUP = X_LIM_BY_DATASET_GROUP_MSE

run_all(ROOT_DIR)


In [ ]:
# ============================================================
# MAE plot settings
# ============================================================
# Y_LIM_MAE controls ONLY the y-axis: Forward model MAE.
#
# Examples:
#   Y_LIM_MAE = None
#   Y_LIM_MAE = (0.0, 0.050)
#   Y_LIM_MAE = (None, 0.050)
#   Y_LIM_MAE = (0.0, None)

Y_LIM_MAE = None
X_LIM_MAE = None

# Optional dataset/group-specific axis limits.
# Keys are: (dataset_name, group)
# group is usually one of: "single", "dual", "multi"
Y_LIM_BY_DATASET_GROUP_MAE = {
    ("Replogle_K562_essential", "single"): (0.26, 0.34),
    ("Replogle_RPE", "single"): (0.25, 0.37),
    ("NormanWeissman2019", "single"): (0.16, 0.23),
    ("NormanWeissman2019", "dual"): (0.17, 0.23),
    ("ChangYe", "single"): (0.13, 0.23),
    ("ZhaoSims2021", "single"): (0.11, 0.20),
}


X_LIM_BY_DATASET_GROUP_MAE = {
    ("Replogle_K562_essential", "single"): (0.27, 0.35),
    ("Replogle_RPE", "single"): (0.25, 0.37),
    ("NormanWeissman2019", "single"): (0.16, 0.23),
    ("NormanWeissman2019", "dual"): (0.17, 0.23),
    ("ChangYe", "single"): (0.18, 0.23),
    ("ZhaoSims2021", "single"): (0.12, 0.18),
}


In [ ]:
# ============================================================
# Run MAE plots
# ============================================================

METRIC_NAME = "mae"
X_COLUMN = "mlp_forward__input_only_mae_mean"
Y_COLUMN = "mlp_forward__model_mae_mean"
REL_COLUMN = "mlp_forward__relative_mae_reduction_mean"

X_LABEL = "Input-only MAE"
Y_LABEL = "Forward model MAE"
PLOT_TITLE = "Forward prediction comparison: MAE"
SAVE_NAME = "forward_mae_scatter_reference_shading"
REL_LABEL = "MAE reduction"

Y_LIM = Y_LIM_MAE
X_LIM = X_LIM_MAE
Y_LIM_BY_DATASET_GROUP = Y_LIM_BY_DATASET_GROUP_MAE
X_LIM_BY_DATASET_GROUP = X_LIM_BY_DATASET_GROUP_MAE

run_all(ROOT_DIR)
